# 01 · Synthea EDA

> **Synthetic data only.** Every dataset used in this notebook is synthetic (Synthea, seed 42) or research-use-only (NIH ChestX-ray14). There is no PHI. None of these models is clinically validated or cleared for patient care.

Exploratory data analysis over a small synthetic Synthea sample using DuckDB and pandas. We profile vitals, look at per-patient trajectories, and sanity-check the value ranges the feature pipeline expects.

## Setup
We read the committed synthetic vitals sample. DuckDB lets us run SQL directly over the CSV; pandas handles the plotting frames.

In [ ]:
from __future__ import annotations
from pathlib import Path
import pandas as pd

try:
    import duckdb
except ImportError:  # duckdb is optional for this EDA
    duckdb = None

SAMPLE = Path('../data/sample/vitals_sample.csv')
df = pd.read_csv(SAMPLE, parse_dates=['ts'])
df.head()

## Profile with DuckDB
Count rows per patient and summarise heart rate and SpO2 — a quick data-quality pass.

In [ ]:
if duckdb is not None:
    con = duckdb.connect()
    con.register('vitals', df)
    summary = con.execute('''
        SELECT patient_id,
               COUNT(*)            AS n_obs,
               ROUND(AVG(heart_rate), 1) AS hr_mean,
               MIN(spo2)           AS spo2_min,
               MAX(resp_rate)      AS rr_max
        FROM vitals GROUP BY patient_id ORDER BY patient_id
    ''').df()
else:
    summary = df.groupby('patient_id').agg(
        n_obs=('heart_rate', 'size'),
        hr_mean=('heart_rate', 'mean'),
        spo2_min=('spo2', 'min'),
        rr_max=('resp_rate', 'max'),
    ).reset_index()
summary

## Vitals trajectories
Plot heart rate over time per patient. Patient `p0001` shows a deteriorating trend (rising HR/RR, falling SpO2) — the kind of signal the sepsis model is meant to pick up.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
for pid, g in df.groupby('patient_id'):
    ax.plot(g['ts'], g['heart_rate'], marker='o', label=pid)
ax.set_xlabel('time'); ax.set_ylabel('heart rate (bpm)')
ax.set_title('Synthetic heart-rate trajectories'); ax.legend()
fig.autofmt_xdate(); plt.tight_layout()

## Range checks vs serving contract
Validate ranges against the `VitalsPoint` bounds used by the serving API so EDA catches contract-violating synthetic rows early.

In [ ]:
checks = {
    'heart_rate': (0, 350), 'spo2': (0, 100), 'resp_rate': (0, 120),
    'temp_c': (20, 45),
}
violations = {
    col: int(((df[col] < lo) | (df[col] > hi)).sum())
    for col, (lo, hi) in checks.items()
}
violations  # all zeros == clean

### Takeaways
* The sample is small but clean and within contract bounds.
* `p0001`/`p0003` trend abnormal; `p0002` is stable.
* Next: turn these raw vitals into the 24×5 model window (notebook 02).